# Architecture IA agentique en entreprise

## Contexte et objectifs

Début 2026, les meilleurs modèles d'IA générative ont passé un seuil critique de capacités : ils permettent désormais de développer des agents IA quasi autonomes capables d'exécuter des tâches significatives en entreprise.

Des expériences sont menées à grande échelle partout dans le monde (Claude Code, OpenClaw ...) à la recherche de l'architecture logicielle et de l'expérience utilisateur idéales pour tirer profit de ces nouvelles capacités.

En mars 2026, il est clair que cette technologie est une révolution dont les entreprises doivent se saisir au plus vite pour rester dans la course, mais quatre problèmes fondamentaux restent à résoudre :

1. La souveraineté technologique : si une grande partie de son fonctionnement est désormais prise en charge par des agents IA, une grande entreprise ne peut pas confier le coeur de son savoir-faire à une plateforme technologique externe qu'elle ne maîtrise pas, elle doit se doter de sa propre intelligence

2. La sécurité des données : les agents IA restent probabilistes et peuvent faire des erreurs, ils représentent une nouvelle surface d'attaque pour des adversaires, il faut identifier les mécanismes qui permettront de les déployer en entreprise sans mettre en danger l'intégrité et la confidentialité des données

3. Le maintien de la supervision et des compétences humaines : pour que l'entreprise puisse garder la maîtrise de son fonctionnement et continuer à produire de la valeur ajoutée à long terme, il faut que ses collaborateurs restent compétents sur les activités différenciantes de leur métier, condition pour qu'ils puissent diriger les agents IA dans la bonne direction

4. L'acceptabilité et la durabilité des agents IA mise en oeuvre dans l'entreprise et dans la société : l'IA agentique doit être déployée au service de l'humain, pour améliorer sa condition, et il faut s'assurer que les ressources de calcul limitées dont le monde disposent sont utilisées pour automatisations qui en valent la peine

## Principes de développement

### Grands modèles de langage (LLMs)

#### Connexion aux plateformes d'inférence de grands modèles de langage (LLMs)

- les plateformes d'inférence des grands modèles de langage (LLMs) sont externes au système d'IA agentique de l'entreprise
- elles peuvent être hébergées et opérées par l'entreprise elle-même ou par un fournisseur tiers, un fournisseur peut proposer plusieurs plateformes d'inférence différentes
- chaque plateforme d'inférence propose un protocole / une API qui lui est propre, avec des fonctionnalités similaires mais tous les fournisseurs ne supportent pas toutes les fonctionnalités
- une plateforme d'inférence peut exposer plusieurs modèles, avec des capacités similaires mais tous les modèles ne supportent pas toutes les fonctionnalités
- la plateforme IA agentique accède à tous les LLMs via une API respectant le standard https://www.openresponses.org, si nécessaire via l'utilisation d'une librairie proxy comme https://github.com/udapy/openresponses-python pour s'adapter au spécificités de protocole de chaque fournisseur
- des mécanismes sont intégrés dans ce proxy pour assurer la disponibilité du service : ré-essayer lorsqu'une erreur s'est produite, gérer les timeouts, équilibrer entre plusieurs services

#### Maîtrise des échanges avec ces plateformes

- le système d'IA agentique de l'entreprise recense les souscriptions aux services d'accès aux LLMs de différents fournisseurs, qui sont associées à un moyen d'authentification (stocké de manière sécurisée)
- un niveau de confiance est associé à chaque souscription, indiquant les données qui peuvent être échangées : confidential, internal, trusted, public
- un mécanisme de filtrage des prompts sortants est mis en place pour vérifier que les données envoyées correspond au niveau de confiance de la souscription
- un mécanisme de filtrage des réponses entrantes est mis en place pour vérifier que les données reçues n'injectent pas des instructions dangereuses
- un système de gestion de priorités, de quotas d'appels et de mesure des coûts est mis en place à l'intérieur du système d'IA agentique de l'entreprise
- il permet de réaliser un suivi et des paramétrages fins liés à l'usage de chaque agent, et est rendu accessible aux composants du système d'IA agentique pour s'auto-réguler
- des fonctionnalités similaires sont certainement implémentées au niveau des plateformes d'inférence elles-mêmes, mais elles agrègent à leur niveau tous les usages d'une souscription
- la plateforme IA agentique d'entreprise permet de piloter et contrôler le partage d'une souscription commune de l'entreprise à un service LLM entre les utilisateurs et agents

#### Plateforme d'inférence LLM pour le développement et le test

- la plateforme d'IA agentique d'entreprise doit pouvoir être développée et testé facilement sans dépendance externe
- une implémentation minimale de plateforme d'inférence LLM est livrée avec la plateforme d'IA agentique de l'entreprise
- elle s'appuie sur une instance vllm avec un modèle open source de petite taille pour les utilisateurs qui possèdent un GPU
- un mode d'emploi pour souscrire au service OpenRouter est proposé pour les utilisateurs qui ne possèdent pas de GPU

#### Découplage avec grand modèle de langage (LLM) utilisé

- un grand modèle de langage est produit par un fournisseur (potentiellement distinct du fournisseur de la plateforme d'inférence) et appartient à une famille de modèles (qui partagent globalement la même architecture et le même jeu de données d'entrainement, mais qui sont de taille et/ou de capacités différentes)
- de nouveaux modèles sont publiés toutes les semaines, chacun possède des forces et faiblesses différents, les politiques commerciales et les conditions d'utilisation des fournisseurs évoluent tous les jours
- une plateforme IA agentique d'entreprise doit pouvoir utiliser différents modèles et en changer en minimisant l'impact sur les agents déjà développés
- un mécanisme optionnel de routage peut être mis en oeuvre par la plateforme IA agentique : les agents ne choisissent pas un modèle spécifique, mais un identifiant correspondant à un niveau de performances et de capacités, et la plateforme IA agentique optimise l'utilisation des souscriptions aux services LLMs disponibles à un instant donné pour fournir les capacités demandées

#### Abstraction des spécificités des modèles par paramétrage

- chaque famille de modèles d'IA agentique est entrainée avec un format de conversation (chat template), des instructions (prompts système, compétences) et des outils de base (manipulation de fichiers, ligne de commande, recherche web, gestion de la session de travail) spécifiques
- même si les LLMs ont une capacité de généralisation qui s'améliore au fil des progrès de la technologie, pour obtenir les meilleurs performances, il reste pertinent d'utiliser le format de conversation, les instructions et outils proposés par l'implémentation de référence d'un système d'IA agentique de chaque fournisseur (Claude Code pour Anthropic, Codex pour OpenAI, Gemini CLI pour Goole, Mistral Vibe pour Mistral, Qwen Code pour Alibaba ...)
- le format de conversation, les prompts et interfaces d'outils standard à utiliser doivent donc faire partie de la configuration associée à chaque famille de modèles
- le système d'IA agentique de l'entreprise représente une session de travail (une conversation avec le LLM) comme une liste de messages structurée selon le modèle conceptuel de données défini par la specification openresponses
- pour formuler une requête au LLM à chaque appel de l'API, il doit convertir cette structure de données en une chaîne de caractères : le format de conversation (chat template) spécifique à chaque modèle indique comment réaliser cette conversion
- de la même manière, lorsque le LLM répond, 
- la construction du prompt système et l'implémentation des outils de la plateforme IA agentique d'entreprise doivent prendre en compte cette configuration spécifique à chaque famille de modèles


### Outils (Tools)

#### Interactions avec le monde